# Smart Parking — Parking Availability Model Research
**North Tel Aviv parking lots — predicting free parking slots**

This notebook conducts an end-to-end data-science study for the Smart Parking
prediction service:

1. Load & explore the synthetic North-Tel-Aviv dataset (5 real lots, 1 year, hourly).
2. Exploratory data analysis (demand by hour / weekday / season / holidays / events).
3. Feature engineering.
4. Time-aware train/test split (honest for time series).
5. Train & compare several regression models — Linear Regression (baseline),
   Decision Tree, **Random Forest**, Gradient Boosting, **XGBoost**, HistGradientBoosting.
6. Evaluate (MAE, RMSE, R²) + cross-validation, feature importance, residuals.
7. Light hyper-parameter tuning of the winner.
8. Conclusion & recommendation for the production model.


In [1]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

ART = Path("artifacts"); ART.mkdir(exist_ok=True)
DATA = Path("../data-gen/parking_north_tlv.csv")
print("Loading", DATA.resolve())

Loading C:\Users\User\bb-work\prediction-service\data-gen\parking_north_tlv.csv


## 1. Load the dataset

In [2]:
df = pd.read_csv(DATA, parse_dates=["timestamp"])
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"period: {df.timestamp.min()}  ..  {df.timestamp.max()}")
print(f"lots: {df.lot_id.nunique()}  |  capacities: {sorted(df.total_slots.unique())}")
df.head()

shape: 43,800 rows x 23 cols
period: 2024-09-01 00:00:00  ..  2025-08-31 23:00:00
lots: 5  |  capacities: [np.int64(120), np.int64(180), np.int64(300), np.int64(420), np.int64(600)]


,timestamp,lot_id,lot_name,lot_type,total_slots,hour_of_day,day_of_week,is_weekend,month,day_of_month,...,holiday_name,is_chol_hamoed,academic_term,is_special_event,event_name,temperature_c,is_rain,occupancy_rate,occupied_slots,free_slots
0,2024-09-01 00:00:00,1,Tel Aviv University (Ramat Aviv),university,420,0,6,0,9,1,...,NaN,0,break,0,NaN,28.8,0,0.2413,101,319
1,2024-09-01 01:00:00,1,Tel Aviv University (Ramat Aviv),university,420,1,6,0,9,1,...,NaN,0,break,0,NaN,29.2,0,0.1043,44,376
2,2024-09-01 02:00:00,1,Tel Aviv University (Ramat Aviv),university,420,2,6,0,9,1,...,NaN,0,break,0,NaN,28.0,0,0.0373,16,404
3,2024-09-01 03:00:00,1,Tel Aviv University (Ramat Aviv),university,420,3,6,0,9,1,...,NaN,0,break,0,NaN,27.3,0,0.0712,30,390
4,2024-09-01 04:00:00,1,Tel Aviv University (Ramat Aviv),university,420,4,6,0,9,1,...,NaN,0,break,0,NaN,26.0,0,0.1071,45,375


In [3]:
df[["total_slots","free_slots","occupancy_rate","temperature_c",
    "is_holiday","is_special_event","is_rain"]].describe().round(2)

,total_slots,free_slots,occupancy_rate,temperature_c,is_holiday,is_special_event,is_rain
count,43800.00,43800.00,43800.00,43800.00,43800.00,43800.00,43800.00
mean,324.00,211.81,0.38,20.95,0.02,0.07,0.12
std,172.23,163.53,0.30,6.55,0.16,0.26,0.32
min,120.00,0.00,0.00,7.00,0.00,0.00,0.00
25%,180.00,76.00,0.10,14.80,0.00,0.00,0.00
50%,300.00,167.00,0.32,20.90,0.00,0.00,0.00
75%,420.00,343.00,0.64,27.10,0.00,0.00,0.00
max,600.00,600.00,1.00,35.10,1.00,1.00,1.00


## 2. Exploratory Data Analysis
We look at the demand drivers the model will need to learn.

In [4]:
# Average occupancy by hour-of-day, per lot type
fig, ax = plt.subplots(figsize=(9,5))
for lt, g in df.groupby("lot_type"):
    g.groupby("hour_of_day").occupancy_rate.mean().plot(ax=ax, marker="o", ms=3, label=lt)
ax.set(title="Average occupancy by hour of day (per lot type)",
       xlabel="hour of day", ylabel="mean occupancy rate")
ax.legend(title="lot type"); fig.tight_layout(); fig.savefig(ART/"eda_occupancy_by_hour.png"); plt.close(fig)
print("saved eda_occupancy_by_hour.png")

saved eda_occupancy_by_hour.png


In [5]:
# Occupancy by day of week (0=Mon .. 4=Fri,5=Sat weekend,6=Sun)
fig, ax = plt.subplots(figsize=(9,5))
piv = df.pivot_table("occupancy_rate","day_of_week","lot_type","mean")
piv.plot(ax=ax, marker="o")
ax.set(title="Average occupancy by day of week (Israeli weekend = Fri/Sat = 4/5)",
       xlabel="day of week (Mon=0)", ylabel="mean occupancy rate")
ax.legend(title="lot type"); fig.tight_layout(); fig.savefig(ART/"eda_by_dow.png"); plt.close(fig)
print("saved eda_by_dow.png")

saved eda_by_dow.png


In [6]:
# Seasonality: monthly mean occupancy
fig, ax = plt.subplots(figsize=(9,5))
df.pivot_table("occupancy_rate","month","lot_type","mean").plot(ax=ax, marker="o")
ax.set(title="Seasonality — mean occupancy by month", xlabel="month", ylabel="mean occupancy rate")
ax.legend(title="lot type"); fig.tight_layout(); fig.savefig(ART/"eda_seasonal.png"); plt.close(fig)
print("saved eda_seasonal.png")

saved eda_seasonal.png


In [7]:
# Effect of holidays and special events (leisure + park lots, where they matter most)
sub = df[df.lot_type.isin(["leisure","park","mall"])]
summary = pd.DataFrame({
    "normal day": sub[(sub.is_holiday==0)&(sub.is_special_event==0)].groupby("lot_type").occupancy_rate.mean(),
    "holiday":    sub[sub.is_holiday==1].groupby("lot_type").occupancy_rate.mean(),
    "special event": sub[sub.is_special_event==1].groupby("lot_type").occupancy_rate.mean(),
})
fig, ax = plt.subplots(figsize=(8,5))
summary.plot.bar(ax=ax)
ax.set(title="Mean occupancy: normal vs holiday vs special event", ylabel="mean occupancy rate", xlabel="lot type")
plt.xticks(rotation=0); fig.tight_layout(); fig.savefig(ART/"eda_events.png"); plt.close(fig)
print(summary.round(3))

          normal day  holiday  special event
lot_type                                    
leisure        0.388    0.429          0.605
mall           0.369    0.121          0.309
park           0.264    0.306          0.480


In [8]:
# Correlation of numeric drivers with occupancy
num = ["hour_of_day","day_of_week","is_weekend","month","total_slots",
       "temperature_c","is_rain","is_holiday","is_chol_hamoed","is_special_event","occupancy_rate"]
fig, ax = plt.subplots(figsize=(8,6.5))
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix (numeric features vs occupancy_rate)")
fig.tight_layout(); fig.savefig(ART/"eda_corr.png"); plt.close(fig)
print("saved eda_corr.png")

saved eda_corr.png


## 3. Feature engineering
Target: **`free_slots`** (what the production service predicts). We keep `total_slots`
so a single model serves lots of every size, add cyclical encodings for hour/month,
and one-hot encode the categorical context.

In [9]:
d = df.copy()
# cyclical encodings
d["hour_sin"] = np.sin(2*np.pi*d.hour_of_day/24); d["hour_cos"] = np.cos(2*np.pi*d.hour_of_day/24)
d["month_sin"] = np.sin(2*np.pi*d.month/12);      d["month_cos"] = np.cos(2*np.pi*d.month/12)
d["event_name"] = d.event_name.fillna("none").replace("", "none")

numeric = ["hour_of_day","day_of_week","is_weekend","month","day_of_month","week_of_year",
           "total_slots","temperature_c","is_rain","is_holiday","is_chol_hamoed","is_special_event",
           "hour_sin","hour_cos","month_sin","month_cos"]
categorical = ["lot_type","season","academic_term","event_name"]

X = pd.concat([d[numeric], pd.get_dummies(d[categorical], prefix=categorical)], axis=1)
y = d["free_slots"]
print(f"feature matrix: {X.shape[1]} features, {X.shape[0]:,} rows")
print("features:", list(X.columns))

feature matrix: 36 features, 43,800 rows
features: ['hour_of_day', 'day_of_week', 'is_weekend', 'month', 'day_of_month', 'week_of_year', 'total_slots', 'temperature_c', 'is_rain', 'is_holiday', 'is_chol_hamoed', 'is_special_event', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'lot_type_leisure', 'lot_type_mall', 'lot_type_park', 'lot_type_residential', 'lot_type_university', 'season_autumn', 'season_spring', 'season_summer', 'season_winter', 'academic_term_break', 'academic_term_exams_a', 'academic_term_exams_b', 'academic_term_semester_a', 'academic_term_semester_b', 'event_name_independence_day', 'event_name_none', 'event_name_port_festival', 'event_name_summer_beach_peak', 'event_name_tlv_marathon', 'event_name_white_night']


## 4. Train / test split — time-aware
For time series, a *chronological* hold-out is more honest than a random split
(it prevents the model from "seeing the future"). We train on the first 80% of
the timeline and test on the most recent 20%.

In [10]:
order = d.timestamp.sort_values().index
split = int(len(order)*0.8)
train_idx, test_idx = order[:split], order[split:]
X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]
print(f"train: {len(train_idx):,}  (<= {d.loc[train_idx,'timestamp'].max()})")
print(f"test : {len(test_idx):,}  (>  {d.loc[test_idx,'timestamp'].min()})")

train: 35,040  (<= 2025-06-19 23:00:00)
test : 8,760  (>  2025-06-20 00:00:00)


## 5. Train & compare models

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.08, max_depth=6,
                            subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                            random_state=RANDOM_STATE, tree_method="hist"),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

cap = df.total_slots.mean()
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
fitted = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    fitted[name] = m
    pred = m.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    cv_mae = -cross_val_score(m, X_train, y_train, cv=cv,
                              scoring="neg_mean_absolute_error", n_jobs=-1).mean()
    rows.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2,
                 "MAE_%cap": 100*mae/cap, "CV_MAE": cv_mae})
    print(f"{name:22s}  MAE={mae:6.2f}  RMSE={rmse:6.2f}  R2={r2:.3f}  CV-MAE={cv_mae:6.2f}")

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
results.round(3)

Linear Regression       MAE= 66.85  RMSE= 86.75  R2=0.729  CV-MAE= 70.19


Decision Tree           MAE= 25.83  RMSE= 44.97  R2=0.927  CV-MAE= 17.19


Random Forest           MAE= 24.63  RMSE= 43.09  R2=0.933  CV-MAE= 15.58


Gradient Boosting       MAE= 37.57  RMSE= 56.91  R2=0.883  CV-MAE= 36.57


XGBoost                 MAE= 23.56  RMSE= 38.10  R2=0.948  CV-MAE= 15.17


HistGradientBoosting    MAE= 23.63  RMSE= 40.41  R2=0.941  CV-MAE= 16.45


,Model,MAE,RMSE,R2,MAE_%cap,CV_MAE
0,XGBoost,23.562,38.101,0.948,7.272,15.167
1,HistGradientBoosting,23.627,40.415,0.941,7.292,16.453
2,Random Forest,24.635,43.091,0.933,7.603,15.584
3,Decision Tree,25.830,44.972,0.927,7.972,17.190
4,Gradient Boosting,37.569,56.909,0.883,11.595,36.566
5,Linear Regression,66.847,86.751,0.729,20.632,70.194


In [12]:
results.to_csv(ART/"metrics.csv", index=False)
fig, ax = plt.subplots(figsize=(9,5))
r = results.set_index("Model")[["MAE","RMSE"]]
r.plot.bar(ax=ax)
ax.set(title="Model comparison on the time-aware test set (lower = better)",
       ylabel="error (free slots)"); plt.xticks(rotation=20, ha="right")
fig.tight_layout(); fig.savefig(ART/"model_comparison.png"); plt.close(fig)
print("saved metrics.csv + model_comparison.png")

saved metrics.csv + model_comparison.png


## 6. Best-model deep dive

In [13]:
best_name = results.iloc[0]["Model"]
best = fitted[best_name]
print("Best model by RMSE:", best_name)

# feature importance (tree models expose feature_importances_)
if hasattr(best, "feature_importances_"):
    imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8,6))
    imp[::-1].plot.barh(ax=ax, color="#2E7D32")
    ax.set(title=f"Top feature importances — {best_name}", xlabel="importance")
    fig.tight_layout(); fig.savefig(ART/"feature_importance.png"); plt.close(fig)
    print(imp.round(3))
else:
    imp = pd.Series(dtype=float)
    print("(best model has no native feature_importances_)")

Best model by RMSE: XGBoost


lot_type_residential        0.473
lot_type_mall               0.146
total_slots                 0.069
lot_type_university         0.061
lot_type_park               0.049
lot_type_leisure            0.035
is_weekend                  0.030
academic_term_break         0.029
hour_of_day                 0.027
hour_sin                    0.014
is_holiday                  0.012
hour_cos                    0.011
academic_term_exams_a       0.009
academic_term_semester_a    0.008
is_chol_hamoed              0.004
dtype: float32


In [14]:
pred = best.predict(X_test)
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(y_test, pred, s=5, alpha=0.25)
lims = [0, max(y_test.max(), pred.max())]
ax.plot(lims, lims, "r--", lw=1)
ax.set(title=f"Predicted vs actual free slots — {best_name}",
       xlabel="actual free slots", ylabel="predicted free slots")
fig.tight_layout(); fig.savefig(ART/"pred_vs_actual.png"); plt.close(fig)

resid = y_test - pred
fig, ax = plt.subplots(figsize=(8,4.5))
sns.histplot(resid, bins=60, kde=True, ax=ax, color="#1565C0")
ax.set(title=f"Residuals (actual - predicted) — {best_name}", xlabel="residual (free slots)")
fig.tight_layout(); fig.savefig(ART/"residuals.png"); plt.close(fig)
print(f"residual mean={resid.mean():.2f}  std={resid.std():.2f}")

residual mean=4.13  std=37.88


## 7. Light hyper-parameter tuning of the winner
A small randomized search to confirm we are near the model's potential.

In [15]:
from sklearn.model_selection import RandomizedSearchCV
tuned_info = {"applied": False}
if best_name == "XGBoost":
    grid = {"n_estimators":[300,500,700], "max_depth":[4,6,8],
            "learning_rate":[0.03,0.05,0.08,0.1], "subsample":[0.8,0.9,1.0]}
    base = XGBRegressor(n_jobs=-1, random_state=RANDOM_STATE, tree_method="hist")
elif best_name == "Random Forest":
    grid = {"n_estimators":[200,400,600], "max_depth":[None,16,24],
            "min_samples_leaf":[1,2,4], "max_features":["sqrt",0.5,1.0]}
    base = RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE)
else:
    grid, base = None, None

if grid:
    search = RandomizedSearchCV(base, grid, n_iter=8, cv=3,
                                scoring="neg_mean_absolute_error",
                                random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    tp = search.best_estimator_.predict(X_test)
    tuned = {"MAE": mean_absolute_error(y_test, tp),
             "RMSE": mean_squared_error(y_test, tp) ** 0.5,
             "R2": r2_score(y_test, tp)}
    tuned_info = {"applied": True, "best_params": search.best_params_, **tuned}
    print("best params:", search.best_params_)
    print(f"tuned  MAE={tuned['MAE']:.2f}  RMSE={tuned['RMSE']:.2f}  R2={tuned['R2']:.3f}")
else:
    print("Winner is not a tree-ensemble requiring tuning; skipping.")

best params: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05}
tuned  MAE=23.64  RMSE=38.57  R2=0.946


## 8. Conclusion & recommendation

In [16]:
best_row = results.iloc[0]
worst_row = results.iloc[-1]
findings = {
    "dataset": {"rows": int(len(df)), "lots": int(df.lot_id.nunique()),
                "period": f"{df.timestamp.min().date()} .. {df.timestamp.max().date()}",
                "capacities": sorted(int(x) for x in df.total_slots.unique())},
    "best_model": best_name,
    "metrics": {r["Model"]: {"MAE": round(r["MAE"],2), "RMSE": round(r["RMSE"],2),
                             "R2": round(r["R2"],3), "MAE_pct_capacity": round(r["MAE_%cap"],2),
                             "CV_MAE": round(r["CV_MAE"],2)}
                for _, r in results.iterrows()},
    "best_metrics": {"MAE": round(best_row.MAE,2), "RMSE": round(best_row.RMSE,2),
                     "R2": round(best_row.R2,3), "MAE_pct_capacity": round(best_row["MAE_%cap"],2)},
    "top_features": list(imp.head(8).index) if len(imp) else [],
    "tuning": tuned_info,
}
Path(ART/"summary.json").write_text(json.dumps(findings, indent=2, default=str), encoding="utf-8")
print(json.dumps(findings, indent=2, default=str))
print(f"\nRECOMMENDATION: deploy {best_name} (MAE {best_row.MAE:.1f} free slots, "
      f"R2 {best_row.R2:.3f}); it clearly beats the {worst_row['Model']} baseline.")

{
  "dataset": {
    "rows": 43800,
    "lots": 5,
    "period": "2024-09-01 .. 2025-08-31",
    "capacities": [
      120,
      180,
      300,
      420,
      600
    ]
  },
  "best_model": "XGBoost",
  "metrics": {
    "XGBoost": {
      "MAE": 23.56,
      "RMSE": 38.1,
      "R2": 0.948,
      "MAE_pct_capacity": 7.27,
      "CV_MAE": 15.17
    },
    "HistGradientBoosting": {
      "MAE": 23.63,
      "RMSE": 40.41,
      "R2": 0.941,
      "MAE_pct_capacity": 7.29,
      "CV_MAE": 16.45
    },
    "Random Forest": {
      "MAE": 24.63,
      "RMSE": 43.09,
      "R2": 0.933,
      "MAE_pct_capacity": 7.6,
      "CV_MAE": 15.58
    },
    "Decision Tree": {
      "MAE": 25.83,
      "RMSE": 44.97,
      "R2": 0.927,
      "MAE_pct_capacity": 7.97,
      "CV_MAE": 17.19
    },
    "Gradient Boosting": {
      "MAE": 37.57,
      "RMSE": 56.91,
      "R2": 0.883,
      "MAE_pct_capacity": 11.6,
      "CV_MAE": 36.57
    },
    "Linear Regression": {
      "MAE": 66.85,
      "RMS